# Direct Laser Control

This notebook uses the Poetry-managed environment. All dependencies from `pyproject.toml` are available.


In [ ]:
# Verify we're using the correct environment
import sys
print(f"Python: {sys.executable}")
print(f"Python version: {sys.version}")

In [ ]:
from femtum_sdk.api.contracts.v1.instruments.health_pb2 import HealthResponse, HealthRequest
from femtum_sdk.api import InstrumentApi

api_host = "localhost:5210"

with InstrumentApi(host=api_host) as api:
    result: HealthResponse = api.health.GetHealth(HealthRequest())
    print(result)


## Fetch Laser Parameters

In [ ]:
from femtum_sdk.api.contracts.v1.dto.femtosecond_laser_pb2 import GatingMode, LaserStatus
from femtum_sdk.api.contracts.v1.instruments.femtosecond_laser_pb2 import (
    GetLaserParametersRequest,
    GetLaserParametersResponse,
)

with InstrumentApi(host=api_host) as api:
    response: GetLaserParametersResponse = api.femtosecond_laser.GetParameters(GetLaserParametersRequest())
    parameters = response.parameters

    print(f"is_enabled: {parameters.is_enabled}")
    print(f"power_level_percent: {parameters.power_level_percent}")
    print(f"gate_control: {GatingMode.Name(parameters.gate_control)} ({parameters.gate_control})")
    print(f"fondamental_pulse_repetition_rate_khz: {parameters.fondamental_pulse_repetition_rate_khz}")
    print(f"pulse_repetition_rate_khz: {parameters.pulse_repetition_rate_khz}")
    print(f"maximum_pulse_repetition_rate_khz: {parameters.maximum_pulse_repetition_rate_khz}")
    print(f"repetition_rate_divider: {parameters.repetition_rate_divider}")
    print(f"pulse_duration_fs: {parameters.pulse_duration_fs}")
    print(f"pulse_dispersion_fs2: {parameters.pulse_dispersion_fs2}")
    print(f"burst_length: {parameters.burst_length}")
    print(f"status: {LaserStatus.Name(parameters.status)} ({parameters.status})")
    print(f"laser_head_temperature_celsius: {parameters.laser_head_temperature_celsius}")
    print(f"laser_in_operation_total_time_seconds: {parameters.laser_in_operation_total_time_seconds}")
    print(f"error: {parameters.error!r}")
    print(f"firmware: {parameters.firmware!r}")
    print(f"serial_number: {parameters.serial_number!r}")



## Assign Parameters

In [ ]:
from femtum_sdk.api.contracts.v1.dto.femtosecond_laser_pb2 import FemtosecondLaserInputParameters
from femtum_sdk.api.contracts.v1.instruments.femtosecond_laser_pb2 import ApplyLaserParametersRequest, ApplyLaserParametersResponse

with InstrumentApi(host=api_host) as api:
    apply_response: ApplyLaserParametersResponse = api.femtosecond_laser.ApplyParameters(ApplyLaserParametersRequest(parameters=FemtosecondLaserInputParameters(
        power_level_percent=0,
        gate_control=GatingMode.GATING_MODE_GATE_ACTIVE_HIGH,
    )))
    print(apply_response.parameters)

    read_response: GetLaserParametersResponse = api.femtosecond_laser.GetParameters(GetLaserParametersRequest())
    parameters = read_response.parameters

    assert parameters.power_level_percent == 0
    assert parameters.gate_control == GatingMode.GATING_MODE_GATE_ACTIVE_HIGH